# Session Exploration

Browse session structure: metadata, parameters, laps, events, and constants.

**Prerequisites:** Complete [01-getting-started.ipynb](01-getting-started.ipynb) first.

In [ ]:
import sys
sys.path.insert(0, '.')
from sqlrace_helpers import (
    init_sqlrace, load_session, session_summary,
    list_parameters, get_laps
)
import pandas as pd

In [ ]:
SESSION_GUID = "<REPLACE WITH YOUR SESSION GUID>"

sm = init_sqlrace()
client_session, session = load_session(sm, SESSION_GUID)
session_summary(session)

## Session metadata

Session items are key-value pairs stored with the session.

In [ ]:
items = session.Items
metadata = {str(items[i].Name): str(items[i].Value)
            for i in range(items.Count)}

if metadata:
    df_meta = pd.DataFrame(metadata.items(), columns=["Key", "Value"])
    display(df_meta)
else:
    print("No session metadata items.")

## Parameters

List all parameters with their identifiers and application groups.

In [ ]:
records = []
for i in range(session.Parameters.Count):
    p = session.Parameters[i]
    records.append({
        "Identifier": str(p.Identifier),
        "Name": str(p.Name),
        "Description": str(p.Description),
        "App Group": str(p.ApplicationGroup) if hasattr(p, 'ApplicationGroup') else "",
    })

df_params = pd.DataFrame(records)
print(f"{len(df_params)} parameter(s)")
display(df_params)

## Laps / Segments

View lap boundaries and timing.

In [ ]:
df_laps = get_laps(session)
if not df_laps.empty:
    print(f"{len(df_laps)} lap(s)")
    display(df_laps)
else:
    print("No laps in session.")

## Constants

Session constants are fixed values stored once (e.g. calibration data).

In [ ]:
constants = session.Constants
const_records = []
for i in range(constants.Count):
    c = constants[i]
    const_records.append({
        "Name": str(c.Name),
        "Value": str(c.Value),
        "Units": str(c.Units) if hasattr(c, 'Units') else "",
    })

if const_records:
    display(pd.DataFrame(const_records))
else:
    print("No constants in session.")

## Markers / Annotations

In [ ]:
markers = session.Markers
marker_records = []
for i in range(markers.Count):
    m = markers[i]
    marker_records.append({
        "Name": str(m.Name),
        "Start (ns)": int(m.StartTime),
        "End (ns)": int(m.EndTime),
        "Type": str(m.Type) if hasattr(m, 'Type') else "",
    })

if marker_records:
    display(pd.DataFrame(marker_records))
else:
    print("No markers in session.")

In [ ]:
client_session.Dispose()
print("Session closed.")